## Load the Model

In [5]:

# solar_efficiency_prediction.py

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor

# Load data
train = pd.read_csv('dataset\\train.csv')
test = pd.read_csv('dataset\\test.csv')

# Data Cleaning
train.fillna({
    'temperature': train['temperature'].mean(),
    'irradiance': train['irradiance'].mean(),
    'panel_age': train['panel_age'].median(),
    'maintenance_count': train['maintenance_count'].median(),
    'soiling_ratio': train['soiling_ratio'].median(),
    'voltage': train['voltage'].mean(),
    'current': train['current'].mean(),
    'module_temperature': train['module_temperature'].mean(),
    'cloud_coverage': train['cloud_coverage'].median(),
    'error_code': train['error_code'].mode()[0],
    'installation_type': train['installation_type'].mode()[0]
}, inplace=True)

test.fillna({
    'temperature': test['temperature'].mean(),
    'irradiance': test['irradiance'].mean(),
    'panel_age': test['panel_age'].median(),
    'maintenance_count': test['maintenance_count'].median(),
    'soiling_ratio': test['soiling_ratio'].median(),
    'voltage': test['voltage'].mean(),
    'current': test['current'].mean(),
    'module_temperature': test['module_temperature'].mean(),
    'cloud_coverage': test['cloud_coverage'].median(),
    'error_code': test['error_code'].mode()[0],
    'installation_type': test['installation_type'].mode()[0]
}, inplace=True)

# Feature Engineering
for df in [train, test]:
    df['power_output'] = df['voltage'] * df['current']
    df['temp_diff'] = df['module_temperature'] - df['temperature']
    df['irradiance_per_cloud'] = df['irradiance'] / (df['cloud_coverage'] + 1)
    
train['power_output'] = train['voltage'] * train['current']
train['temp_diff'] = train['module_temperature'] - train['temperature']
train['irradiance_per_cloud'] = train['irradiance'] / (train['cloud_coverage'] + 1)

    
test['power_output'] = test['voltage'] * test['current']
test['temp_diff'] = test['module_temperature'] - test['temperature']
test['irradiance_per_cloud'] = test['irradiance'] / (test['cloud_coverage'] + 1)


# Encode categorical features
cat_cols = ['string_id', 'error_code', 'installation_type']
for col in cat_cols:
    train[col] = train[col].astype('category').cat.codes
    test[col] = test[col].astype('category').cat.codes

# Feature selection
drop_cols = ['id', 'efficiency']
X = train.drop(columns=drop_cols)
y = train['efficiency']
X_test = test.drop(columns=['id'])

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)


In [22]:
# Round all float columns in training and test datasets to 4 decimals
float_cols = X_train.select_dtypes(include='float').columns

X_train[float_cols] = X_train[float_cols].round(4)
X_val[float_cols] = X_val[float_cols].round(4)
X_test[float_cols] = X_test[float_cols].round(4)

In [36]:
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    X_train[col] = X_train[col].round(4)
    # X_train[col]
    
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    X_test[col] = X_test[col].round(4)
    # X_train[col]
    
for col in ['humidity', 'wind_speed', 'pressure']: 
    # Step 1: Convert 'humidity' column to float
    X_val[col] = pd.to_numeric(X_val[col], errors='coerce')

    # Step 2: Round the values to 4 decimal places
    X_val[col] = X_val[col].round(4)
    # X_train[col]

5894      3.0728
3728     36.1907
8958     73.9350
7671     33.3638
5999     61.3078
          ...   
11284     3.1201
11964    83.3350
5390     97.9866
860      74.0606
15795    56.3914
Name: humidity, Length: 16000, dtype: float64

In [37]:

# Model
model = CatBoostRegressor(iterations=150, learning_rate=0.1, depth=6, random_seed=42, verbose=False)
model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)


In [18]:
# # Step 1: Check all columns for non-numeric values
# for col in X_train.columns:
#     if X_train[col].dtype == 'object':
#         print(f"Non-numeric values in '{col}':")
#         print(X_train[col].unique())


Non-numeric values in 'humidity':
['3.0727649717016248' '36.19069387932499' '73.93500547165834' ...
 '97.98664107084582' '74.06063797969237' '56.39137460128162']
Non-numeric values in 'wind_speed':
['14.882793001988924' '3.3268778675965027' '13.134006734283506' ...
 '11.410031110186647' '3.5226585234190964' '3.0990640688180826']
Non-numeric values in 'pressure':
['1008.4450250185085' '1022.411506297662' '1009.3768610135843' ...
 '1027.5773552401697' '1001.3618987923786' '1026.661553929723']


In [19]:
# # Replace bad strings with NaN across the full test dataset
# X_train.replace("badval", np.nan, inplace=True)

# # Then, fill with appropriate strategy
# # Use median for numeric, mode for categorical
# num_cols = X_test.select_dtypes(include=['float64', 'int64']).columns
# cat_cols = X_test.select_dtypes(include='object').columns

# print(cat_cols)
# print(num_cols)


Index(['humidity', 'wind_speed', 'pressure'], dtype='object')
Index(['temperature', 'irradiance', 'panel_age', 'maintenance_count',
       'soiling_ratio', 'voltage', 'current', 'module_temperature',
       'cloud_coverage', 'power_output', 'temp_diff', 'irradiance_per_cloud'],
      dtype='object')


In [38]:

# for col in cat_cols:
#     X_test[col] = pd.to_numeric(X_test[col], errors='coerce')
#     X_test[col].fillna(X_train[col].median(), inplace=True)

# for col in cat_cols:
#     X_test[col].fillna(X_train[col].mode()[0], inplace=True)


In [39]:
X_train

,temperature,irradiance,humidity,panel_age,maintenance_count,soiling_ratio,voltage,current,module_temperature,cloud_coverage,wind_speed,pressure,string_id,error_code,installation_type,power_output,temp_diff,irradiance_per_cloud
5894,26.7958,506.4186,3.0728,33.1335,2.0,0.4973,14.1078,1.0627,33.5309,56.3129,14.8828,1008.4450,2,0,2,14.9917,6.7351,8.8360
3728,25.0772,570.5220,36.1907,9.1141,1.0,0.5076,0.0000,0.3447,39.7320,81.1741,3.3269,1022.4115,3,0,1,0.0000,14.6548,6.9428
8958,32.7002,454.9752,73.9350,1.4014,4.0,0.8779,12.0523,0.8134,40.1731,46.8084,13.1340,1009.3769,3,1,2,9.8039,7.4729,9.5166
7671,43.3006,808.4606,33.3638,8.7681,2.0,0.5042,5.3303,3.2522,50.4316,49.7041,11.0275,1010.4104,2,0,2,17.3354,7.1309,15.9447
5999,19.8420,154.3096,61.3078,17.4977,1.0,0.8394,0.0000,1.8810,20.5645,84.8135,4.4632,1006.7739,3,0,2,0.0000,0.7225,1.7982
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,37.7532,456.2477,3.1201,17.4977,7.0,0.6906,34.4881,2.0959,45.4146,59.0494,4.4668,1002.5267,2,0,0,72.2822,7.6613,7.5979
11964,56.7182,302.1620,83.3350,33.1524,4.0,0.6977,12.2043,2.0781,55.3807,90.7496,3.3472,1021.5981,0,1,1,25.3619,-1.3375,3.2933
5390,36.9065,901.1025,97.9866,17.3730,4.0,0.7272,25.2611,2.4249,44.0937,84.4953,11.4100,1027.5774,1,0,2,61.2563,7.1873,10.5398
860,37.7476,254.4951,74.0606,7.3531,4.0,0.8060,11.1051,0.3237,45.7887,83.1775,3.5227,1001.3619,3,0,1,3.5950,8.0411,3.0233


In [40]:

# Validation score
y_pred = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred))
score = (1 - rmse) * 100
print(f"RMSE: {rmse:.4f}, Score: {score:.2f}")


RMSE: 0.1061, Score: 89.39


In [52]:
import datetime

# Using datetime.date.today()
today_date = datetime.date.today()
print(today_date)

# Using datetime.datetime.now()
current_datetime = datetime.datetime.now()
print(current_datetime)

from datetime import datetime

# Get current date and time
now = datetime.now()

# Format it as desired (e.g., YYYYMMDD_HHMMSS)
timestamp = now.strftime("%H%M%S")
print(f"submission_{timestamp}")

2025-06-07
2025-06-07 16:25:47.864585
submission_162547


In [ ]:
# Final prediction


preds = model.predict(X_test)
submission = pd.DataFrame({'id': test['id'], 'efficiency': preds})

submission.to_csv(f'submission_{timestamp}.csv', index=False)
